In [13]:
import pystormtracker as pst
import numba
import numpy as np
import xarray as xr

import sys
import os

import matplotlib.pyplot as plt
import matplotlib
from tqdm import tqdm
import seaborn as sns
from scipy import stats
import cmocean
import pandas as pd
import cartopy.crs as ccrs
from matplotlib.colors import TwoSlopeNorm


In [14]:
ds = xr.open_dataset("/Users/asgerthormann/Documents/uni/master/master_code/master_thesis/data/single_levels_6h/data_stream-oper_stepType-instant.nc")

In [15]:
ds

<xarray.Dataset> Size: 1GB
Dimensions:     (valid_time: 244, latitude: 281, longitude: 601)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 2kB 1999-11-01 ... 1999-12-31T18:...
    expver      (valid_time) <U4 4kB ...
  * latitude    (latitude) float64 2kB 90.0 89.75 89.5 89.25 ... 20.5 20.25 20.0
  * longitude   (longitude) float64 5kB -90.0 -89.75 -89.5 ... 59.5 59.75 60.0
    number      int64 8B ...
Data variables:
    u10         (valid_time, latitude, longitude) float32 165MB ...
    v10         (valid_time, latitude, longitude) float32 165MB ...
    d2m         (valid_time, latitude, longitude) float32 165MB ...
    t2m         (valid_time, latitude, longitude) float32 165MB ...
    msl         (valid_time, latitude, longitude) float32 165MB ...
    sst         (valid_time, latitude, longitude) float32 165MB ...
    sp          (valid_time, latitude, longitude) float32 165MB ...
    i10fg       (valid_time, latitude, longitude) float32 165MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-09-02T14:48 GRIB to CDM+CF via cfgrib-0.9.1...

In [16]:
tracker = pst.HodgesTracker()

In [17]:
tracker.track?

Signature:
tracker.track(
    data: 'TrackingInput',
    variable: 'str',
    *,
    start_time: 'TimeInput | None' = None,
    end_time: 'TimeInput | None' = None,
    time_step: 'np.timedelta64 | None' = None,
    detection_mode: 'DetectionMode' = 'auto',
    object_threshold: 'float | None' = None,
    engine: 'str | None' = None,
    **kwargs: 'object',
) -> 'Tracks'
Docstring: <no docstring>
File:      ~/mamba/envs/master_env/lib/python3.12/site-packages/pystormtracker/hodges/tracker.py
Type:      method

In [30]:
from pystormtracker.hodges import constants
print(vars(constants))


{'__name__': 'pystormtracker.hodges.constants', '__doc__': None, '__package__': 'pystormtracker.hodges', '__loader__': <_frozen_importlib_external.SourceFileLoader object at 0x301a5de50>, '__spec__': ModuleSpec(name='pystormtracker.hodges.constants', loader=<_frozen_importlib_external.SourceFileLoader object at 0x301a5de50>, origin='/Users/asgerthormann/mamba/envs/master_env/lib/python3.12/site-packages/pystormtracker/hodges/constants.py'), '__file__': '/Users/asgerthormann/mamba/envs/master_env/lib/python3.12/site-packages/pystormtracker/hodges/constants.py', '__cached__': '/Users/asgerthormann/mamba/envs/master_env/lib/python3.12/site-packages/pystormtracker/hodges/__pycache__/constants.cpython-312.pyc', '__builtins__': {'__name__': 'builtins', '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module i

In [31]:
INPUT_FILE = "/Users/asgerthormann/Documents/uni/master/master_code/master_thesis/data/single_levels_6h/data_stream-oper_stepType-instant.nc"       # your buffered NetCDF/GRIB subset
VARIABLE = "msl"               # or "vo" for 850 hPa relative vorticity
DETECTION_MODE = "min"         # "max" for vorticity

FRAME_WORKERS = 8
MGE_WORKERS = 4   # cap near expected segment count
 
tracker = pst.HodgesTracker(
    feature_refinement="bspline",   # rectangular TRACK/SMOOPY-compatible path
    dmax=6.5,                       # deg, TRACK default upper displacement bound
    phimax=0.5,
    w1=0.2,
    w2=0.8,
    min_track_points=3,
    segment_frames=62,              # leave default; fine for this record length
    backend="dask",
    frame_workers=FRAME_WORKERS,
    sht_threads=None,               # auto
    mge_workers=MGE_WORKERS,
    lmin=5,
    lmax=42,        # the classic Hodges/TRACK T5-42 band
    taper_points=5, # see note below on regional boundaries
)



print(vars(tracker))          # dict of every set attribute
 

{'w1': 0.2, 'w2': 0.8, 'dmax': 6.5, 'phimax': 1.0, 'mge_max_iterations': 3, 'min_track_points': 3, 'projection': 'global', 'stereo_grid_spacing_km': 100.0, 'extent': None, 'lmin': 5, 'lmax': 42, 'taper_points': 5, 'spectral_taper': 1.0, 'search_window_size': 3, 'min_object_grid_points': 1, 'feature_refinement': 'bspline', 'group_adjacent_extrema': False, 'exclude_boundary_extrema': False, 'bspline_smoothing': 0.0, 'track_smoopy_optimization_scale': 1.0, 'bspline_max_iterations': 100, 'bspline_gradient_tolerance': 1e-05, 'backend': 'dask', 'frame_workers': 8, 'sht_threads': None, 'mge_workers': 4, 'segment_frames': 62, 'dmax_zones': array([[  0. , 360. , -90. , -20. ,   6.5],
       [  0. , 360. , -20. ,  20. ,   3. ],
       [  0. , 360. ,  20. ,  90. ,   6.5]]), 'adaptive_smoothness': array([[1. , 2. , 5. , 8. ],
       [1. , 0.3, 0.1, 0. ]]), 'missing_frame_parameters': array([[6.5, 1. ]])}


In [32]:
ds = xr.open_dataset(INPUT_FILE)
ds = ds.rename({"valid_time": "time"})   # only if needed
if "expver" in ds.dims:
    ds = ds.sel(expver=1)                 # pick final ERA5 (drop ERA5T), or combine_first

In [33]:
print(ds)

<xarray.Dataset> Size: 1GB
Dimensions:    (time: 244, latitude: 281, longitude: 601)
Coordinates:
  * time       (time) datetime64[ns] 2kB 1999-11-01 ... 1999-12-31T18:00:00
    expver     (time) <U4 4kB ...
  * latitude   (latitude) float64 2kB 90.0 89.75 89.5 89.25 ... 20.5 20.25 20.0
  * longitude  (longitude) float64 5kB -90.0 -89.75 -89.5 ... 59.5 59.75 60.0
    number     int64 8B ...
Data variables:
    u10        (time, latitude, longitude) float32 165MB ...
    v10        (time, latitude, longitude) float32 165MB ...
    d2m        (time, latitude, longitude) float32 165MB ...
    t2m        (time, latitude, longitude) float32 165MB ...
    msl        (time, latitude, longitude) float32 165MB ...
    sst        (time, latitude, longitude) float32 165MB ...
    sp         (time, latitude, longitude) float32 165MB ...
    i10fg      (time, latitude, longitude) float32 165MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-

In [34]:
tracks = tracker.track(
    ds,
    VARIABLE,
    detection_mode=DETECTION_MODE,
)

In [35]:
tracks[0]

In [36]:
tracks.write("tracks.trackjson")